# Theory
### 1. Tensors and Autograd  
##### 1.1 What is a tensor?

Very short answer: A tensor in PyTorch = like a NumPy array, but with superpowers:
- It works on CPU and GPU.
- It can keep track of operations to compute gradients automatically.
- It supports many extra operations used in deep learning.

Example: converting between NumPy and PyTorch:  
```python
import numpy as np
import torch

np_array = np.array([1, 2, 3, 4])

# NumPy → Tensor
tensor = torch.from_numpy(np_array)  # shares memory
# or
tensor = torch.tensor(np_array)      # copies data

# Tensor → NumPy
new_np_array = tensor.numpy()

print(type(np_array))    # <class 'numpy.ndarray'>
print(type(tensor))      # <class 'torch.Tensor'>
print(type(new_np_array))# <class 'numpy.ndarray'>
```

Key differences vs NumPy:
- NumPy: only CPU, no autograd.

PyTorch tensor:
- can live on GPU (.to("cuda")),
- can track gradients if requires_grad=True.

##### 1.2 requires_grad=True: tell PyTorch “I want gradients”
When you create a tensor:
```python
x = torch.tensor(3.0, requires_grad=True)
```

You are saying:  
> “Track all operations that use x, because later I want the derivative of something with respect to x.”  
If requires_grad=False (default), PyTorch doesn’t track ops and you can’t call .backward() through those tensors.  

##### 1.3 Computation graph, .backward(), and .grad
```python
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2          # y = x²

y.backward()        # compute dy/dx
print(x.grad)       # 6.0 (since d(x²)/dx = 2x, at x=3 → 6)
```

What’s going on:  
1.	Computation graph  
- PyTorch internally builds a graph:  
> x --(square)--> y  
- Each node knows how it was computed (which operation).

2.	.backward()  
- y.backward() = “Go backwards through the graph and compute the gradient of y with respect to all leaf tensors that have requires_grad=True.”
- Here, the only such tensor is x, so it computes dy/dx.

3.	.grad  
- x.grad stores the gradient $\frac{\partial y}{\partial x}$ evaluated at the current value of x.

Important correction to your intuition:
PyTorch isn’t literally “undoing” operations; it’s using the chain rule for derivatives along the graph of operations. But you get what I mean.

##### 1.4 two-parameter model
```python
import torch

# Input and target
x = torch.tensor(1.0)
y = torch.tensor(2.0)

# Parameters we want to learn
theta1 = torch.tensor(3.0, requires_grad=True)
theta2 = torch.tensor(4.0, requires_grad=True)

# Model: y_pred = x * theta1 * theta2
y_pred = x * theta1 * theta2

# Loss: (y_pred - y)^2
loss = (y_pred - y) ** 2

loss.backward()

print(theta1.grad)  # dLoss/dtheta1
print(theta2.grad)  # dLoss/dtheta2
```

Your algebra:
- $a = \theta_1 \theta_2$
- $b = a x = \theta_1 \theta_2 x$
- $c = b - y = \theta_1 \theta_2 x - y$
- $L = c^2 = (\theta_1 \theta_2 x - y)^2$

loss.backward() computes, via chain rule:
- $\frac{\partial L}{\partial \theta_1}$
- $\frac{\partial L}{\partial \theta_2}$

Those values go into:
- theta1.grad
- theta2.grad

Takeaway:
- .backward() computes gradients for all tensors with requires_grad=True that influence the loss.
- You don’t manually write derivatives; PyTorch does it.

---

### 2. Linear Layers (nn.Linear)

##### 2.1 What is a linear layer?
A linear layer does this math:  
$\text{output} = x W^T + b$

Where:
- x = input features (row vector of size in_features)
- W = weight matrix of shape (out_features, in_features)
- b = bias vector of shape (out_features,)
- output = vector of size out_features

You create it in PyTorch with:
```python
import torch
import torch.nn as nn

linear_layer = nn.Linear(in_features=3, out_features=2)

print("Weight:", linear_layer.weight)  # shape (2, 3)
print("Bias:", linear_layer.bias)      # shape (2,)
```

Magic part:
```python
linear_layer = nn.Linear(3, 2)
```

##### 2.2 Manual vs nn.Linear computation
```python
import torch
import torch.nn as nn

linear_layer = nn.Linear(in_features=3, out_features=2)

input_tensor = torch.tensor([[1.0, 2.0, 3.0]])  # shape (1, 3)

weight = linear_layer.weight          # shape (2, 3)
bias = linear_layer.bias              # shape (2,)

# Manual computation:
output_manual = input_tensor @ weight.T + bias

# Using the layer:
output_layer = linear_layer(input_tensor)

print(output_manual)
print(output_layer)
# They are (numerically) the same
```

> nn.Linear = “multiply by W, add b” = the basic ingredient of neural nets.

---

### 3. Activation Functions
Linear layer alone = just a linear transformation.  
Stacking only linear layers = still overall linear. Boring, weak.  

> You need non-linear activation functions between linear layers to model complex stuff.

##### 3.1 ReLU
Definition:  
$\text{ReLU}(x) = \max(0, x)$
- Negative values → 0
- Positive values → unchanged
```python
x = torch.tensor([-1.0, 0.0, 1.0, 2.0])
relu_output = torch.relu(x)
print(relu_output)  # tensor([0., 0., 1., 2.])
```

##### 3.2 Sigmoid
Definition:  

$\sigma(x) = \frac{1}{1 + e^{-x}}$

> Maps any real number → (0, 1)
```python
x = torch.tensor([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0])
sigmoid_output = torch.sigmoid(x)
print(sigmoid_output)
```

##### 3.3 Softmax and logits
Logits = raw outputs of the network (no activation yet).  
- They can be any real numbers.
- Not constrained, don’t sum to anything.

Softmax turns logits into probabilities:  
Given a vector $z = [z_1, z_2, ..., z_K]$:
$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$

Properties:
- All outputs > 0
- They sum to 1
→ valid probability distribution.
```python
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0]
])

softmax_output = torch.softmax(x, dim=1)  # softmax across columns
print(softmax_output)
```

> Last layer: nn.Linear(...) → logits → torch.softmax(logits, dim=1) (or use CrossEntropyLoss which applies softmax internally).

---

### 4. Modules and Models (nn.Module)

##### 4.1 What is a Module?
In PyTorch, a Module is the basic building block:  
- Linear layers, convolution layers, activation layers, whole models, etc. = all subclasses of nn.Module.

Why?  
A Module:
- stores parameters (weights, biases),
- knows how to move to GPU/CPU,
- knows how to save/load state,
- can be nested inside other Modules.

##### 4.2 Defining your own model
You create a model by subclassing nn.Module:  
```python
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()            # important: init base nn.Module stuff
        self.layer1 = nn.Linear(3, 2) # define layers here

    def forward(self, x):
        return self.layer1(x)         # define how data flows
```

Two key parts:
- 1.	__init__:  
    - Here you define layers and components (things with parameters).
	- Example: self.layer1 = nn.Linear(3, 2).

- 2.	forward(self, x):
    - Here you define the computation

> When you later call model(x), PyTorch runs forward internally.

> ##### 4.3 Stacking modules
You can stack layers in forward:
```python
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(3, 4)
        self.layer2 = nn.Linear(4, 2)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.layer2(x)
        return x
```

- Or use nn.Sequential (shortcut):
```python
model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 2)
)
```

We'll see this better in a minute

---

### 6. Two-layer fully connected model

##### 6.1 What the model is
```python
class SimpleNeuralNet(nn.Module):
    def __init__(self):
        super(SimpleNeuralNet, self).__init__()
        self.layer1 = nn.Linear(3, 4)  # input: 3, hidden: 4
        self.layer2 = nn.Linear(4, 2)  # hidden: 4, output: 2

    def forward(self, x):
        x = self.layer1(x)       # Linear: (batch, 3) → (batch, 4)
        x = torch.relu(x)        # Non-linearity
        x = self.layer2(x)       # Linear: (batch, 4) → (batch, 2)
        return x                  # output logits (2 numbers per sample)
```

- Example usage:
```python
model = SimpleNeuralNet()

# two samples, each with three features
input_tensor = torch.tensor([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0]
])

output = model(input_tensor)
print("Model output:", output)   # shape (2, 2)
```

Mental model:
- First layer: pulls 3 input features → creates 4 hidden features.
- ReLU: kills negative values → adds non-linearity.
- Second layer: maps those 4 features to 2 outputs (e.g., 2 classes).

---

### 7. A simple 3-layer network with Softmax

##### 7.1 Model architecture

You wrote something like:
- input → Linear → ReLU → Linear → Softmax
```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 1st Linear layer (input → hidden)
        self.fc1 = nn.Linear(in_features=4, out_features=3)
        # 2nd Linear layer (hidden → output)
        self.fc2 = nn.Linear(in_features=3, out_features=2)

    def forward(self, x):
        x = self.fc1(x)          # (batch, 4) → (batch, 3)
        x = F.relu(x)            # non-linearity
        x = self.fc2(x)          # (batch, 3) → (batch, 2) logits
        x = F.softmax(x, dim=1)  # turn logits into probabilities
        return x
```
- Input example (batch of 2 samples, 4 features each):
```python
model = SimpleNet()

x = torch.tensor([
    [1.0, 2.0, 0.5, 1.5],
    [3.0, 4.0, 2.5, 3.5]
])

output = model(x)
print(output)   # shape (2, 2), each row sums to 1
```

Data flow (super-basic brain mode):  
1.	x → fc1  
- Multiply by first weight matrix, add bias → hidden features.

2.	→ ReLU  
- Zero out negatives → non-linearity.

3.	→ fc2  
- Another linear transformation → logits (raw scores).

4.	→ Softmax  
- Convert logits to probabilities (non-negative, sum to 1).

NOTE (for training):  
> For multi-class classification, with nn.CrossEntropyLoss(), you usually do not put softmax inside forward. You return the logits and let CrossEntropyLoss apply softmax internally.


---

### 8. Loss functions

##### 8.1 What is a loss function?
- It measures how wrong your model is.
- Training = try to minimize the loss by changing weights.

> These are loss functions, not just “evaluation methods”.
> - You use them during training to compute a number (the loss) and its gradient.
> - BCEWithLogitsLoss and CrossEntropyLoss really do the sigmoid/softmax inside themselves

#### 8.2 Loss vs “evaluation method”
Think of 2 things:  
1.	Loss function (training objective)
- Example: nn.MSELoss, nn.BCEWithLogitsLoss, nn.CrossEntropyLoss.
- You call it during training:
```python
logits = model(x)
loss = loss_fn(logits, y)
loss.backward()
optimizer.step()
```

It:
- computes a scalar “how wrong am I?”
- and PyTorch uses it to compute gradients for backprop.

2.	Evaluation metrics  
- Example: accuracy, F1, precision, recall, etc.
- You compute them just to see “how good am I?”, no gradients needed.
```python
with torch.no_grad():
    logits = model(x_val)
    preds = logits.argmax(dim=1)
    acc = (preds == y_val).float().mean()
```

##### 8.3 nn.BCEWithLogitsLoss – what does it actually do?
*Combines sigmoid + binary cross-entropy in one numerically stable function” what do you mean? is it just for evaluation or when I recall it it also performs sigmoid + binary cross-entropy?*

It really performs both steps inside:
- Conceptually, for each sample:
	- Take your model output z (logit, any real number).
	- Compute p = sigmoid(z).
	- Compute binary cross-entropy:
$\text{BCE}(p, y) = -[y \log p + (1-y)\log(1-p)]$
	- Average over the batch.

But it does this in one optimized, numerically stable formula (not literally calling torch.sigmoid then log etc), to avoid overflows/underflows.

##### **COMPLETE EXAMPLE**
```python
import torch
import torch.nn as nn
import torch.optim as optim

# 1) Make a tiny model for binary classification
class TinyBinaryNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(2, 1)   # 2 inputs -> 1 output (LOGIT)

    def forward(self, x):
        # IMPORTANT: NO sigmoid here
        return self.fc(x)           # returns raw scores (logits)

model = TinyBinaryNet()

# 2) Loss function for binary classification (logits in, labels 0/1)
loss_fn = nn.BCEWithLogitsLoss()

# 3) Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.1)

# 4) Fake data (2 samples, 2 features each)
x = torch.tensor([[0.0, 0.0],
                  [1.0, 1.0]])   # inputs
y = torch.tensor([[0.0],
                  [1.0]])        # targets: first is class 0, second is class 1

# === ONE TRAINING STEP ===
optimizer.zero_grad()          # clear old gradients

logits = model(x)              # forward pass → raw scores (no sigmoid)
loss = loss_fn(logits, y)      # compute "how wrong" we are
loss.backward()                # compute gradients
optimizer.step()               # update weights

print("Logits:", logits)
print("Loss:", loss.item())
```

##### > **What is logits = model(x)?**
- model(x) runs your network.
- Last layer is Linear(..., 1) with no sigmoid.
- So the output is just some random numbers, e.g.:
```text
tensor([[-0.4231],
        [ 0.8123]])
```
--> These are logits:
- They can be any real number (negative, positive, huge, tiny).
- They are NOT probabilities yet.

If you wanted the probability that y=1, you’d do:
```python
probs = torch.sigmoid(logits)
```
> ...**BUT** during training with BCEWithLogitsLoss, you don’t do that yourself.
> The loss does this internally.

> **What is loss = loss_fn(logits, y) actually doing?**
“Combines sigmoid + binary cross-entropy”
Translate that to “idiot mode”:
1.	Takes the logits (raw scores) from the model.  
2.	Inside itself, it does:  
- “First I pretend to apply sigmoid to turn logits into probabilities.”
- “Then I compute how far those probabilities are from the true labels 0/1 using binary cross-entropy.”
3.	It gives you one number = the average error over the batch.  

So conceptually:
```python
logits = model(x)        # raw scores
loss  = BCEWithLogits(logits, y)
```

is like:
```python
logits = model(x)                  # raw scores
probs  = torch.sigmoid(logits)     # (this step is hidden inside the loss)
# then compute BCE between probs and y (also inside the loss)
```

> You don’t see sigmoid, but it’s happening under the hood.

So:  
> BCEWithLogitsLoss = “I take your raw scores, apply sigmoid, and compute BCE, all in one go.”

##### What does loss.backward() do?
You now have loss, a single scalar number, e.g. 0.72.
```python
loss.backward()
```
This means:  
> “PyTorch, go back through all the operations that produced this loss and compute the derivative of the loss with respect to each weight and bias in the model.”

Result: Every parameter in model.parameters() that has requires_grad=True gets a .grad filled in.
> You don’t care about the math, PyTorch does the chain rule stuff.

##### What does optimizer.step() do?
> You have gradients stored in param.grad for each parameter.  
Now:  
```python
optimizer.step()
```
does:
> “Update every weight = weight – learning_rate * gradient”
for all parameters in the optimizer.

> ### So the workflow is:
> ### 1.	logits = model(x)
> ##### → “What do I predict?”
> ### 2.	loss = loss_fn(logits, y)
> ##### → “How wrong am I?”
> ### 3.	loss.backward()
> ##### → “How should I move each weight to be less wrong?”
> ### 4.	optimizer.step()
> ##### → “Actually move the weights a bit.”

---

### 9. Optimizers and gradient descent

##### 9.1 Concept
Gradient descent idea:
$\text{new\_weights} = \text{old\_weights} - \eta \cdot \nabla_\theta L$
- $\eta$ = learning rate
- $\nabla_\theta L$ = gradient of the loss w.r.t. parameters $\theta$

In words:  
> “Move weights a little bit in the direction that reduces the loss.”
> Optimizers in PyTorch are the things that actually perform this update.


##### 9.2 Adam optimizer example
```python
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
```
Where:  
- model.parameters() = all weights and biases inside your model that have requires_grad=True.
	- model.parameters() returns all nn.Parameter objects in the model (weights, biases, etc.),
	- some of them can have requires_grad=True, some False.
	- The optimizer can see them all, but only the ones with requires_grad=True will actually get gradients and be updated.


- lr = learning rate:
	- too big → training diverges,
	- too small → training is painfully slow.

Adam = “fancy” gradient descent:
- Keeps track of moving averages of gradients and squared gradients.

##### 9.3 Example: check what model.parameters() contains
import torch
import torch.nn as nn
import torch.optim as optim

# Define a tiny model
```python
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 4)   # has weight & bias
        self.fc2 = nn.Linear(4, 2)   # has weight & bias

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = TinyNet()

print("=== Named parameters and requires_grad ===")
for name, param in model.named_parameters():
    print(name, "| shape:", param.shape, "| requires_grad:", param.requires_grad)
```
And this is the output:
```python
fc1.weight | shape: torch.Size([4, 3]) | requires_grad: True
fc1.bias   | shape: torch.Size([4])    | requires_grad: True
fc2.weight | shape: torch.Size([2, 4]) | requires_grad: True
fc2.bias   | shape: torch.Size([2])    | requires_grad: True
```

- model.parameters() is a generator over these 4 tensors (all the weights & biases).
- By default, all of them have requires_grad=True.

> Now let’s freeze one layer.

##### Example: freezing some params (requires_grad=False)
```python
# Freeze fc1: we don't want to train this layer
for param in model.fc1.parameters():
    param.requires_grad = False

print("\n=== After freezing fc1 ===")
for name, param in model.named_parameters():
    print(name, "| shape:", param.shape, "| requires_grad:", param.requires_grad)
```
This is the output:  
```python
fc1.weight | shape: torch.Size([4, 3]) | requires_grad: False
fc1.bias   | shape: torch.Size([4])    | requires_grad: False
fc2.weight | shape: torch.Size([2, 4]) | requires_grad: True
fc2.bias   | shape: torch.Size([2])    | requires_grad: True
```

Important point:
- fc1.weight and fc1.bias are still in model.parameters().
- But their requires_grad=False, so:
	- they will not get gradients (.grad stays None),
	- they will not be updated in any meaningful way by training.

---

### 10. One gradient descent step (in practice)
The classic 5 sub-steps:

1.	Clear old gradients  
- Every parameter has a .grad field.
- Gradients accumulate by default (PyTorch adds, doesn’t overwrite).
- So before computing new gradients:
```python
optimizer.zero_grad()
```

2.	Forward pass  
- Compute model predictions:
```python
output = model(x)  # x = batch of inputs
```

3.	Compute loss
- Compare output to ground-truth labels:
```python
loss = loss_fn(output, y)
```

4.	Backward pass
- Compute gradients of loss w.r.t. all parameters:
```python
loss.backward()
```

5.	Update parameters
- Use the optimizer to apply the gradient step:
```python
optimizer.step()
```

**SO:**
```python
optimizer.zero_grad()      # 1. Clear gradients
output = model(x)          # 2. Forward
loss = loss_fn(output, y)  # 3. Loss
loss.backward()            # 4. Backward
optimizer.step()           # 5. Update
```

> That’s **one** update step.

---

### 11. Training loop, epochs, and SGD

##### 11.1 Why one update is not enough

One update = tiny nudge in the right direction.  
You need many updates to:  
- slowly push the loss down,
- slowly adjust weights to fit the data.

##### 11.2 What would “ideal” gradient descent look like?
You have a training set with N samples.  
Let’s call it:  
$\{(x_1, y_1), \dots, (x_N, y_N)\}$  

> The “ideal” gradient descent step is:
> Use all N samples at once to compute the loss and its gradient.

In code (conceptually):
```python
# BAD idea in practice if N is big
optimizer.zero_grad()
preds = model(X_all)            # X_all has ALL samples
loss = loss_fn(preds, y_all)    # loss over ALL samples
loss.backward()                 # gradient over ALL samples
optimizer.step()                # update once
```

Problems:
- Needs a ton of memory.
- Each step is expensive (uses the whole dataset).
- You still need many steps → painfully slow.

> So: computing the gradient on the entire dataset for every single step is what is “too heavy”.

##### 11.3 What do we actually do? → Mini-batch SGD
> We split the dataset into smaller pieces: batches.

Say:
- Dataset size N = 10,000
- Batch size B = 100

Then each batch contains 100 samples.
You have:
$\text{steps per epoch} = N / B = 10,000 / 100 = 100 \text{ steps}$

> Now, instead of:
> “Use all 10,000 samples to do 1 update”

> we do:
> “Use 100 samples to do 1 update, and repeat this 100 times per epoch”.

```python
for x_batch, y_batch in train_loader:   # each batch is 100 samples here
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = loss_fn(preds, y_batch)
    loss.backward()
    optimizer.step()
```

Each iteration of this inner loop is:
- one approximate gradient (on 100 samples),
- one update step.
This is mini-batch SGD.  

> Key idea:
- A batch is a subset used to approximate the “true” gradient, the gradient for the step is computed on a subset of all the dataset (ex. 100 rows instead of the whole 10.000)
- we use subsets (batches) instead of the full dataset.

##### 11.4 So what the hell is an epoch then?
> Now, imagine you run that inner loop over all batches in the dataset once.
> That means:
> - You have seen every training sample exactly once (in some batch).
> - You have done 100 GPU/CPU-friendly updates instead of 1 giant, heavy update.

> That whole pass over the dataset is called an **epoch**.
```python
for epoch in range(num_epochs):             # repeat the whole process multiple times
    for x_batch, y_batch in train_loader:   # go through all batches
        optimizer.zero_grad()
        preds = model(x_batch)
        loss = loss_fn(preds, y_batch)
        loss.backward()
        optimizer.step()
```
means:
- Inner loop (for x_batch, y_batch in train_loader):
	- Go through all batches (each batch = subset).
	- **This is one epoch**

- Outer loop (for epoch in range(num_epochs)):
	- See all the batches multiple times (num_epochs), so see all the dataset divided in batches multiple times (num_epochs).
	- So the model sees the training data again and again, refining weights each time.

### 11.5.Analogy (because why not)
Imagine you have to learn a 300-page book.  
Full-batch gradient descent:  
- Read all 300 pages,
- Then adjust your understanding once.
- Very heavy, long, exhausting, but your update is based on all pages at once.

Mini-batch SGD with epoch:  
- Read 10 pages → adjust your understanding a bit,
- Read next 10 pages → adjust again,
- …
- After 30 chunks, you’ve seen all 300 pages = 1 epoch.

You still read the whole book, but:
- You updated your “parameters” (understanding) many times along the way,
- Each update was based on a smaller chunk (less heavy per step).

> Then you might reread the book (another epoch), noticing new details, fixing misconceptions, etc.

##### 11.6 Why multiple epochs?
Even after seeing the whole dataset once (1 epoch), the model is usually:
- Not well-trained yet, just slightly better.
- So you go through the dataset again (another epoch).
- Each time:
	- weights start slightly better than before,
	- updates (per batch) refine them more.

So:
> - Inner loop = many SGD steps (one per batch).
> - One pass over all batches = 1 epoch.
> - Outer loop = do multiple epochs to train better.

##### 11.7 PyTorch utilities / tools

You’ll often see:
> torch.utils.data.Dataset
--> A class to represent a dataset (how to get length and individual elements).

> torch.utils.data.DataLoader
--> Wraps a dataset and gives you batches:
```python
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
```

Then the training loop:
```python
for epoch in range(num_epochs):
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(x_batch)
        loss = loss_fn(preds, y_batch)
        loss.backward()
        optimizer.step()
```

---

---

# Cool complete example

Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

Build the model

In [ ]:
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        
        # Input: 3 features, Output: 2 features
        self.layer1 = nn.Linear(3, 5)  # First layer, 3 inputs to 5 outputs
        self.relu = nn.ReLU()           # ReLU activation function
        self.layer2 = nn.Linear(5, 1)   # Second layer, 5 inputs to 1 output (binary output)

    def forward(self, x):
        # Pass the input through the first layer, apply ReLU, then the second layer
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

Initialize the model, the loss function and the optimizer

In [ ]:
# Create an instance of the model
model = SimpleModel()

# Binary Cross-Entropy Loss (for binary classification)
loss_fn = nn.BCEWithLogitsLoss()

# Optimizer (we'll use Adam with a learning rate of 0.001)
optimizer = optim.Adam(model.parameters(), lr=0.001)

Get the input data  

x_train = training values  
y_train = the corresponding training labels

In [ ]:
# Creating dummy data
x_train = torch.tensor([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [3.0, 6.0, 9.0], [4.0, 8.0, 12.0]])  # Input features (4 samples, 3 features each)
y_train = torch.tensor([[1.0], [0.0], [1.0], [0.0]])  # Target labels (0 or 1)

Training loop

In [ ]:
num_epochs = 10  # Number of complete passes over the data
batch_size = 2   # Number of samples in each batch

for epoch in range(num_epochs):
    # Loop over the data in batches
    for i in range(0, len(x_train), batch_size):
        # Get the batch of data
        x_batch = x_train[i:i + batch_size]
        y_batch = y_train[i:i + batch_size]
        
        # Step 1: Zero the gradients (reset previous gradients)
        optimizer.zero_grad()
        
        # Step 2: Forward pass (calculate predictions)
        logits = model(x_batch)  # Output: raw scores (logits)
        
        # Step 3: Calculate the loss
        loss = loss_fn(logits, y_batch)  # Compare logits to true labels (y_batch)
        
        # Step 4: Backward pass (compute gradients)
        loss.backward()  # This computes the gradient of loss wrt the model parameters
        
        # Step 5: Update model parameters (weights and biases)
        optimizer.step()  # Use the gradients to update model parameters
        
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}')

### Still not clear what are these numbers
I know this is the loss functions, but how is is computed, which are the true labels, ecc.?

##### Step 1: Forward Pass (Getting Predictions)
The model takes the input features and produces logits (raw scores):  
```python
logits = model(x_batch)
```

Assume the model gives the following logits:
```python
logits = tensor([[ 0.5], [-0.3]])  # These are the raw outputs from the model
```

##### mmmhhhhh ... can we describe the whole pipeline? How do I end up with these values?
The input passes through this:  
- 1. First layer: 
```python 
output = x_batch @ W1 + b1
```
- Which returns:
```python
layer1_output = tensor([[a1, a2, a3, a4, a5], 
                        [b1, b2, b3, b4, b5]])
```
- 2. ReLu activation:
```python
ReLU(x) = max(0, x)
```
- Which returns:
```python
relu_output = tensor([[max(0, a1), max(0, a2), max(0, a3), max(0, a4), max(0, a5)], 
                      [max(0, b1), max(0, b2), max(0, b3), max(0, b4), max(0, b5)]])
```
- 3. Second layer:
```python
output = relu_output @ W2 + b2
```

- Which returns:
```python
logits = tensor([[ 0.5], 
                 [-0.3]])
```

##### Step 2: Apply Sigmoid Activation (for Probabilities)
Since you’re using BCEWithLogitsLoss, PyTorch will automatically apply the sigmoid function to the logits. The sigmoid function turns the raw logits into probabilities (values between 0 and 1):
```python
sigmoid(logits) = 1 / (1 + exp(-logits))
```

For the logits above:
- logit = 0.5 → Sigmoid(0.5) ≈ 0.622
- logit = -0.3 → Sigmoid(-0.3) ≈ 0.425

So, the predicted probabilities would be:
```python
pred_probs = tensor([[0.62245935], [0.42555748]])
```

These are the model’s predictions for each sample in the batch. The model thinks:
- The first sample has a 62.2% chance of being class 1.
- The second sample has a 42.6% chance of being class 1.

##### Step 3: Calculate Loss (Using Binary Cross-Entropy)
The loss function calculates the difference between the predicted probabilities and the true labels (targets) using binary cross-entropy. The formula for binary cross-entropy is:
$BCE = - [y * log(pred) + (1 - y) * log(1 - pred)]$

Where:
- y is the true label (target).
- pred is the predicted probability (after sigmoid).




SO, since for the first sample we have:
- True label y = 1.0 (the true label is class 1).
- Predicted probability pred = 0.62245935 (the model predicted 62.2% chance of class 1).

If we apply the formula:
```python
BCE_1 = - [1 * log(0.62245935) + (1 - 1) * log(1 - 0.62245935)]
       ≈ - log(0.62245935) ≈ 0.474
```

While for the second sample:
- True label y = 0.0 (the true label is class 0).
- Predicted probability pred = 0.42555748 (the model predicted 42.6% chance of class 1).

Using the formula:
```python
BCE_2 = - [0 * log(0.42555748) + (1 - 0) * log(1 - 0.42555748)]
       ≈ - log(0.57444252) ≈ 0.554
```

##### Step 4: Calculate Average Loss for the Batch
Now, the loss function calculates the average loss over all samples in the batch. For this batch of 2 samples:
```python
loss = (BCE_1 + BCE_2) / 2
     ≈ (0.474 + 0.554) / 2
     ≈ 0.514
```

##### Step 5: Backpropagation and Update
Once the loss is calculated, the model performs backpropagation (loss.backward()), which computes the gradients of the loss with respect to the model parameters (weights and biases). Then, the optimizer updates the model’s parameters using these gradients (optimizer.step()).

